# 04. Binary Search on Answer

Sometimes there is **no sorted array**. Instead, the **answer itself** lies in a range (like speed, capacity, days), and you can test "does value `x` work?". If the test result flips only once (`False False True True`), binary search the answer.

**Learning order:** IDEA → CHECKLIST → KOKO BANANAS → SHIP PACKAGES → MAXIMIZE TEMPLATE → MAX CANDIES → INTERVIEW THINKING

## 1. Real-life idea

"What is the **minimum speed** I must eat bananas so that I finish in `h` hours?"

You can't directly compute it, but you can easily **test** a speed: "if I eat 5 per hour, how many hours do I need?" If a speed works, any **faster** speed also works. That is monotonic (`False False True True True`), so binary search the speed.

Other examples: minimum truck capacity to ship in D days, minimum time to finish all jobs, maximum minimum distance between cows.

## 2. Checklist: is it binary search on answer?

1. Question says "**minimum** X such that ..." or "**maximum** X such that ...".
2. You can write `feasible(x)` that returns True/False in O(n).
3. If `x` is feasible, then a **larger** (for minimize) or **smaller** (for maximize) x is also feasible → monotonic.
4. You can name a **range** `[lo, hi]` for the answer.

**Cost:** `O(n × log(range))`.

## 3. Koko Eating Bananas

Piles of bananas, `h` hours. Each hour Koko eats from one pile at speed `k`. Minimum `k` to finish in `h` hours?

- Range: `1` to `max(piles)`.
- `feasible(k)`: total hours = sum of `ceil(pile / k)` ≤ `h`.
- Minimize → find the **first True**.

In [1]:
def min_eating_speed(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        hours = sum((p + mid - 1) // mid for p in piles)   # ceil(p / mid)
        if hours <= h:
            hi = mid          # works, try slower
        else:
            lo = mid + 1      # too slow
    return lo

print(min_eating_speed([3, 6, 7, 11], 8))        # 4
print(min_eating_speed([30, 11, 23, 4, 20], 5))  # 30
print(min_eating_speed([30, 11, 23, 4, 20], 6))  # 23

4
30
23


## 4. Capacity To Ship Packages Within D Days

Packages must go in order. Minimum ship capacity so all ship within `days`?

- Range: `max(weights)` (must carry the heaviest) to `sum(weights)` (everything in one day).
- `feasible(cap)`: count days greedily; start a new day when the next package does not fit.

In [2]:
def ship_within_days(weights, days):
    def feasible(cap):
        d, load = 1, 0
        for w in weights:
            if load + w > cap:
                d += 1
                load = 0
            load += w
        return d <= days

    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5))   # 15
print(ship_within_days([3, 2, 2, 4, 1, 4], 3))                # 6

15
6


## 5. Maximize template (last True)

For "**maximum** x such that feasible(x)", feasible is `True True True False False`. Find the **last True**.

Use the **upper middle** `mid = (lo + hi + 1) // 2`, and `lo = mid` when True. If you use the lower middle with `lo = mid`, the loop gets stuck when `hi = lo + 1`.

In [3]:
def max_candies(candies, k):
    # largest pile size x so that at least k children each get x candies
    lo, hi = 0, max(candies)
    while lo < hi:
        mid = (lo + hi + 1) // 2           # upper middle
        if sum(c // mid for c in candies) >= k:
            lo = mid                       # feasible, try bigger
        else:
            hi = mid - 1
    return lo

print(max_candies([5, 8, 6], 3))   # 5
print(max_candies([2, 5], 11))     # 0

5
0


## 6. Interview thinking

**Say:** "The answer lies in `[lo, hi]`. I can check a candidate in O(n). The check is monotonic, so I binary search the answer. Total O(n log range)."

**Common mistakes:** wrong `lo`/`hi` range (e.g. `lo = 1` when the answer can be 0, or `lo` less than `max(weights)`), using lower middle in the maximize version (infinite loop), a `feasible` that is not monotonic, forgetting ceil division, off-by-one in the days counter (starts at 1).

**Related:** *Split Array Largest Sum* is the same as the shipping problem. *Aggressive Cows* is the maximize template.

## Quick Revision

- "min X such that ..." / "max X such that ..." + a testable `feasible(x)` → binary search on answer.
- Minimize: first True, `hi = mid` if feasible else `lo = mid + 1`.
- Maximize: last True, `mid = (lo+hi+1)//2`, `lo = mid` if feasible else `hi = mid - 1`.
- Range: smallest possible answer to largest possible answer.
- Time O(n × log range).